# Coffee17 B0 SAM–AdamW reference V1 — input selector revision V2
Exactly five new development training runs, seed 42, fixed rho=0.05, 50 epochs. Reuse audited B0 recovery validation results. No B0 retraining, no C2 training, no outer-test inference. Attach original Coffee17 and the exact coffee17-b0-c2-recovery-analysis-v1.zip (as a Kaggle dataset input) OR the extracted/saved recovery folder. Old checkpoints are not required. For resume also attach previous SAM saved output. Keep all new checkpoints. Scientific pin unchanged.

In [ ]:
import os, sys, subprocess, tempfile, json, zipfile, hashlib
from pathlib import Path
import torch
PIN="a539a776b7960c9622fd59280d17252a0c27401a"
INPUT=Path("/kaggle/input")
WORK=Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir()
if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU")
print("TRAINING: B0_SAM ONLY | 5 FOLDS | SEED 42")
print("OUTER TEST: LOCKED | B0 BASELINE: NO RETRAIN")
print("CHECKPOINT DELETION: DISABLED")
print("PIN:",PIN)
REPO=Path(tempfile.mkdtemp(prefix="coffee17-sam-reference-src-"))/"repo"
subprocess.run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-classification.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",PIN],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements/preprocessing-study.txt")],check=True)
ENV=dict(os.environ)
ENV["PYTHONPATH"]=str(REPO/"src")+os.pathsep+ENV.get("PYTHONPATH","")
sys.path.insert(0,str(REPO/"src"))
RUNNER=REPO/"src/bilinear_lmmd/experiments/run_sam_reference_v1.py"
compile(RUNNER.read_text(),str(RUNNER),"exec")
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.experiments.run_c2_internal_oof_v1 import AUTH_SHA
from bilinear_lmmd.core.reproducibility import sha256_file

# Accept the exact audited compact recovery ZIP or an extracted/saved recovery folder.
# Evidence inputs may be mounted under /kaggle/input/datasets OR notebooks.
# Original Coffee17 image discovery below remains separate.
RECOVERY_ZIP_SHA="07e9e796cc6357bfa178f3e46ec85c86a372dd2a2f893dfe1840207209b4ee6f"
RECOVERY_ZIP_SIZE=530051
archives=sorted(p for p in INPUT.rglob("*.zip") if p.is_file())
exact_zips=[p for p in archives if p.stat().st_size==RECOVERY_ZIP_SIZE and sha256_file(p)==RECOVERY_ZIP_SHA]
if exact_zips:
    archive=exact_zips[0]
    RECOVERY=Path(tempfile.mkdtemp(prefix="coffee17-sam-audited-baseline-"))
    with zipfile.ZipFile(archive) as z:
        names=z.namelist()
        if len(names)!=83 or len(set(names))!=83 or z.testzip() is not None:
            raise RuntimeError("Recovery ZIP integrity failed")
        for name in names:
            rel=Path(name)
            if rel.is_absolute() or ".." in rel.parts:
                raise RuntimeError("Unsafe ZIP member")
        z.extractall(RECOVERY)
    if sha256_file(RECOVERY/"recovery_authority.json")!=AUTH_SHA:
        raise RuntimeError("Recovery authority hash mismatch after extraction")
    print("AUDITED RECOVERY ZIP:",archive)
    print("RECOVERY ZIP SHA256: MATCH")
else:
    candidates=sorted(p for p in INPUT.rglob("recovery_authority.json") if p.is_file())
    matches=[p.parent for p in candidates if sha256_file(p)==AUTH_SHA and all(
        (p.parent/"experiments"/f"fold_{f}"/"seed42"/"B0"/"completed.json").is_file()
        for f in range(1,6))]
    if not matches:
        print("KAGGLE INPUT MOUNTS:",[str(p) for p in sorted(INPUT.iterdir())])
        print("RECOVERY AUTHORITIES FOUND:",[str(p) for p in candidates])
        print("ZIP INPUTS FOUND:",[str(p) for p in archives])
        raise RuntimeError("Add the exact coffee17-b0-c2-recovery-analysis-v1.zip as a Kaggle Dataset input, or attach its extracted audited folder. No old checkpoints required. Expected ZIP SHA256: "+RECOVERY_ZIP_SHA)
    if len(matches)>1:
        raise RuntimeError("Multiple audited extracted recovery folders; attach one or use the exact recovery ZIP")
    RECOVERY=matches[0]
    print("AUDITED RECOVERY FOLDER:",RECOVERY)
print("RECOVERY EVIDENCE ROOT:",RECOVERY)
# Same dataset-tree selector as recovery revision V2.
SLUG="coffee-green-bean-with-17-defects-original"
DATASETS_INPUT=INPUT/"datasets"
preferred=[DATASETS_INPUT/"sujitraarw"/SLUG,INPUT/SLUG]
present=[p for p in preferred if p.is_dir()]
if len(present)>1:
    raise RuntimeError(f"Multiple original dataset mounts: {present}")
if present:
    DATASET=present[0]
    samples=discover_directory_samples(DATASET)
else:
    valid=[]
    if DATASETS_INPUT.is_dir():
        for root in [DATASETS_INPUT]+sorted(p for p in DATASETS_INPUT.rglob("*") if p.is_dir()):
            try: samples=discover_directory_samples(root)
            except (ValueError,FileNotFoundError): continue
            if len(samples)==17 and sum(map(len,samples.values()))==979: valid.append((root,samples))
    deepest=[(root,samples) for root,samples in valid if not any(other!=root and root in other.parents for other,_ in valid)]
    if len(deepest)!=1:
        print("KAGGLE INPUT MOUNTS:",[str(p) for p in sorted(INPUT.iterdir())])
        raise RuntimeError("Attach one original Coffee17 dataset (not notebook outputs). Valid roots: "+str([str(x[0]) for x in deepest]))
    DATASET,samples=deepest[0]
assert len(samples)==17 and sum(map(len,samples.values()))==979
print("DATASET:",DATASET,"979/17")
OUT=WORK/"coffee17-b0-sam-reference-v1"
SCRATCH=Path("/tmp/coffee17_b0_sam_reference_v1_data")
import shutil
# A previous SAM project may be attached for resume. Never overwrite an existing project.
prior=sorted(p.parent for p in INPUT.rglob("sam_authority.json") if p.is_file() and "datasets" not in p.relative_to(INPUT).parts)
if not OUT.exists() and len(prior)>1:
    raise RuntimeError("Attach at most one prior SAM saved output")
if not OUT.exists() and len(prior)==1:
    shutil.copytree(prior[0],OUT)
    print("RESTORED SAM PROJECT:",prior[0])
OUT.mkdir(parents=True,exist_ok=True)



In [ ]:
# Training only for B0_SAM; all source and baseline hashes plus numerical tests
# must pass before training. A completed fold is skipped after artifact checks.
cmd=[sys.executable,"-u","-m","bilinear_lmmd.experiments.run_sam_reference_v1",
     "--recovery",str(RECOVERY),"--dataset",str(DATASET),"--output",str(OUT),
     "--scratch",str(SCRATCH),"--required-commit",PIN,"--authorize-sam-reference"]
with (OUT/"sam_reference.log").open("a",encoding="utf-8") as log:
    proc=subprocess.Popen(cmd,cwd=REPO,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end="",flush=True)
        log.write(line);log.flush()
    if proc.wait()!=0:
        raise RuntimeError("SAM stopped. Existing artifacts preserved; inspect log.")


In [ ]:
# Packaging only: never trains, moves or deletes checkpoint files.
SUMMARY=OUT/"analysis/sam_reference_v1.json"
assert SUMMARY.is_file(),"All five SAM runs must finish first"
summary=json.loads(SUMMARY.read_text())
assert summary["training_runs"]==5 and summary["baseline_retrained"] is False and summary["outer_test_accessed"] is False
for f in range(1,6):
    d=OUT/"experiments"/f"fold_{f}"/"seed42"/"B0_SAM"
    marker=json.loads((d/"completed.json").read_text())
    for name,digest in marker["files"].items():
        assert sha256_file(d/name)==digest,(f,name)
    assert (d/"best.pt").is_file() and (d/"last.pt").is_file()
ZIP=WORK/"coffee17-b0-sam-reference-analysis-v1.zip"
files=sorted(p for p in OUT.rglob("*") if p.is_file() and p.suffix in {".json",".csv",".yaml"} and not p.name.endswith(".tmp"))
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        assert set(z.namelist())=={str(p.relative_to(OUT)) for p in files}
        for p in files:
            assert hashlib.sha256(z.read(str(p.relative_to(OUT)))).hexdigest()==sha256_file(p),"Existing ZIP differs; preserve it"
else:
    tmp=ZIP.with_suffix(".tmp")
    with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in files:z.write(p,str(p.relative_to(OUT)))
    with zipfile.ZipFile(tmp) as z:assert z.testzip() is None
    tmp.replace(ZIP)
print("10 BEST/LAST CHECKPOINTS VERIFIED AND RETAINED")
print("BASELINE RETRAINED: NO")
print("OUTER TEST: LOCKED")
print("DECISION:",summary["decision"])
print(json.dumps(summary["means"],indent=2))
print("UPLOAD:",ZIP)
print("KEEP SAVED PROJECT:",OUT)
